# Pew Section

In [1]:
from pathlib import Path
from src.pew_worker import PewWorker

# The notebook runs from the project root
PROJECT_ROOT = Path.cwd()

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\KING\Desktop\lab2_tidying_cleaning_imputation_outlier


In [2]:
# Locate the Pew dataset in the data directory
pew_files = sorted((PROJECT_ROOT / "data").rglob("Pew*.csv"))

print("Pew files found:", len(pew_files))
for file in pew_files:
    print(file.name)

Pew files found: 1
Pew Research Center Global Attitudes Spring 2025 Dataset CSV.csv


In [3]:
# Create the PEW worker and load the dataset
pew = PewWorker(pew_files[0])
pew_df = pew.load()

print("Dataset shape:", pew_df.shape)
display(pew_df.head())

Dataset shape: (28333, 483)


,survey,country,weight,id,econ_sit,satisfied_democracy,polsys_satisfied,fav_us,fav_china,fav_eu,...,region_skorea,region_spain,region_sweden,region_turkey,region_uk,urbanicity,qdate_s,qdate_e,intdate_aus,phone_sample
0,1202501,2,0.592005,8001432,2,2.0,NaN,4,3,2,...,NaN,NaN,NaN,NaN,NaN,NaN,3/23/2025,3/23/2025,NaN,1.0
1,1202501,2,0.446814,8003472,4,3.0,NaN,4,4,2,...,NaN,NaN,NaN,NaN,NaN,NaN,3/11/2025,3/11/2025,NaN,1.0
2,1202501,2,1.120110,8006468,2,1.0,NaN,3,8,2,...,NaN,NaN,NaN,NaN,NaN,NaN,4/1/2025,4/1/2025,NaN,1.0
3,1202501,2,0.695317,8006970,2,1.0,NaN,4,8,1,...,NaN,NaN,NaN,NaN,NaN,NaN,3/10/2025,3/10/2025,NaN,1.0
4,1202501,2,0.369352,8008601,2,2.0,NaN,4,3,1,...,NaN,NaN,NaN,NaN,NaN,NaN,3/23/2025,3/23/2025,NaN,1.0


In [4]:
# Reshape the income data
pew_melted = pew.melt_income()

print("Reshaped dataset:", pew_melted.shape)
display(pew_melted.head())

# Summarize nonblank responses by income variable
pew_summary = pew.summary()

print("Summary columns:", pew_summary.columns.tolist())
display(pew_summary.head(10))

Reshaped dataset: (29438, 7)


,id,country,weight,income_variable,income_code,measure,country_name
0,2002686,23,0.356581,d_income_argentina,9.0,income,argentina
1,2002725,23,2.364549,d_income_argentina,4.0,income,argentina
2,2002731,23,1.512416,d_income_argentina,NaN,income,argentina
3,2002734,23,0.621898,d_income_argentina,7.0,income,argentina
4,2002918,23,0.472910,d_income_argentina,6.0,income,argentina


Summary columns: ['country_name', 'income', 'income2']


,country_name,income,income2
0,argentina,902,99
1,brazil,1011,42
2,canada,904,120
3,france,954,47
4,germany,935,71
5,greece,982,28
6,hungary,948,72
7,india,3347,198
8,indonesia,1019,3
9,israel,861,137


 # Billboard Section

### Setup

In [5]:
from pathlib import Path
from src.billboard_worker import BillboardWorker

PROJECT_ROOT = Path.cwd()
billboard_file = PROJECT_ROOT / "data" / "billboard.csv"

print("Billboard file:", billboard_file, "| exists:", billboard_file.is_file())

Billboard file: c:\Users\KING\Desktop\lab2_tidying_cleaning_imputation_outlier\data\billboard.csv | exists: True


### Load the raw data

In [6]:
billboard = BillboardWorker(billboard_file)
billboard_df = billboard.load()          # or load(source="neon") for the Neon table

print("Raw shape:", billboard_df.shape)
display(billboard_df.head())

Raw shape: (317, 77)


,artist,track,duration,genre,date entered,date peaked,weeks to peak,weeks on chart,worst position,best position,...,56th week,57th week,58th week,59th week,60th week,61st week,62nd week,63rd week,64th week,65th week
0,Destiny's Child,Independent Women Part I,218,Rock,9/23/2000,11/18/2000,8,28,78,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Santana,"Maria, Maria",258,Rock,2/12/2000,4/8/2000,8,26,48,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Savage Garden,I Knew I Loved You,247,Rock,10/23/1999,1/29/2000,14,33,71,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Madonna,Music,225,Rock,8/12/2000,9/16/2000,5,24,44,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Aguilera, Christina",Come On Over Baby (All I Want Is You),218,Rock,8/5/2000,10/14/2000,10,21,57,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Melt the weekly columns into week and rank

In [7]:
# One row per song per week (wide → long)
bb_melted = billboard.melt_weeks()

print("Melted shape:", bb_melted.shape)
display(bb_melted[["artist", "track", "week", "rank"]].head())

Melted shape: (20605, 14)


,artist,track,week,rank
0,Destiny's Child,Independent Women Part I,1st week,78.0
1,Santana,"Maria, Maria",1st week,15.0
2,Savage Garden,I Knew I Loved You,1st week,71.0
3,Madonna,Music,1st week,41.0
4,"Aguilera, Christina",Come On Over Baby (All I Want Is You),1st week,57.0


In [8]:
bb_melted = billboard.clean_week_column()

print("Week dtype:", bb_melted["week"].dtype)
print("Week range:", bb_melted["week"].min(), "to", bb_melted["week"].max())

Week dtype: int64
Week range: 1 to 65


In [9]:
before = len(bb_melted)
bb_tidy = billboard.drop_empty_ranks()

print(f"Rows: {before:,} → {len(bb_tidy):,} ({before - len(bb_tidy):,} empty weeks removed)")

Rows: 20,605 → 5,307 (15,298 empty weeks removed)


In [10]:
bb_tidy = billboard.add_chart_date()

display(bb_tidy[["artist", "track", "date entered", "week", "chart_date", "rank"]].head(10))

,artist,track,date entered,week,chart_date,rank
0,Destiny's Child,Independent Women Part I,9/23/2000,1,2000-09-23,78.0
1,Santana,"Maria, Maria",2/12/2000,1,2000-02-12,15.0
2,Savage Garden,I Knew I Loved You,10/23/1999,1,1999-10-23,71.0
3,Madonna,Music,8/12/2000,1,2000-08-12,41.0
4,"Aguilera, Christina",Come On Over Baby (All I Want Is You),8/5/2000,1,2000-08-05,57.0
5,Janet,Doesn't Really Matter,6/17/2000,1,2000-06-17,59.0
6,Destiny's Child,Say My Name,12/25/1999,1,1999-12-25,83.0
7,"Iglesias, Enrique",Be With You,4/1/2000,1,2000-04-01,63.0
8,Sisqo,Incomplete,6/24/2000,1,2000-06-24,77.0
9,Lonestar,Amazed,6/5/1999,1,1999-06-05,81.0


In [ ]:
fig = billboard.plot_rankings()        
fig.show()

### Cars section

In [ ]:
from src.cars_worker import CarsWorker

cars = CarsWorker()         

In [ ]:
# Uploads cars.csv (as text) and a 1,000-row EPA sample to Neon
# display(cars.upload_sources())

,rows_uploaded
cars_raw,407
cars_public_epa,1000


In [15]:
cars_df = cars.load()
print("Loaded:", cars_df.shape)

cars_df = cars.drop_irrelevant_rows()
print("After dropping the STRING;DOUBLE;... row:", cars_df.shape)
display(cars_df.head())

Loaded: (407, 9)
After dropping the STRING;DOUBLE;... row: (406, 9)


,Car,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model,Origin
0,Chevrolet Chevelle Malibu,NaN,8,307.0,130.0,3504.,12.0,70,US
1,Buick Skylark 320,15.0,8,350.0,NaN,3693.,11.5,70,US
2,Plymouth Satellite,NaN,8,318.0,150.0,3436.,11.0,70,US
3,AMC Rebel SST,16.0,8,NaN,150.0,NaN,12.0,70,US
4,Ford Torino,17.0,8,302.0,140.0,3449.,10.5,70,US


In [16]:
# Must run BEFORE convert_types, which turns the zeros into NaN
print("Zero values per column:")
display(cars.zero_report())

cars_df = cars.convert_types()   # MPG/Horsepower zeros → NaN
display(cars_df.dtypes)
display(cars.missing_report())

Zero values per column:


MPG             8
Cylinders       0
Displacement    0
Horsepower      6
Weight          0
Acceleration    0
Model           0
dtype: int64

Car                  str
MPG              float64
Cylinders          int64
Displacement     float64
Horsepower       float64
Weight           float64
Acceleration     float64
Model              int64
Origin          category
dtype: object

,missing,missing_%,valid
Car,0,0.00,406
MPG,11,2.71,395
Cylinders,0,0.00,406
Displacement,1,0.25,405
Horsepower,8,1.97,398
Weight,1,0.25,405
Acceleration,0,0.00,406
Model,0,0.00,406
Origin,0,0.00,406


In [17]:
cars_df = cars.merge_public_data()
display(cars.source_summary())
display(cars.missing_report())

,rows,first_year,last_year
source,,,
epa_public,1000,84,89
original,406,70,82


,missing,missing_%,valid
Car,0,0.00,1406
MPG,11,0.78,1395
Cylinders,0,0.00,1406
Displacement,1,0.07,1405
Horsepower,1008,71.69,398
Weight,1001,71.19,405
Acceleration,1000,71.12,406
Model,0,0.00,1406
Origin,0,0.00,1406
source,0,0.00,1406


In [18]:
dups = cars.duplicates_report()
print("Rows with an exact duplicate:", len(dups))
display(dups)

cars_df = cars.drop_duplicates()
print("After dropping duplicates:", cars_df.shape)

Rows with an exact duplicate: 94


,Car,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model,Origin,source
410,Aston Martin Saloon/Vantage/Volante,8.0,8.0,323.4,NaN,NaN,NaN,85,Europe,epa_public
554,Aston Martin Saloon/Vantage/Volante,8.0,8.0,323.4,NaN,NaN,NaN,85,Europe,epa_public
770,BMW 6 Series,16.0,6.0,207.5,NaN,NaN,NaN,87,Europe,epa_public
771,BMW 6 Series,16.0,6.0,207.5,NaN,NaN,NaN,87,Europe,epa_public
798,Buick Somerset/Skylark,23.0,4.0,152.6,NaN,NaN,NaN,87,US,epa_public
...,...,...,...,...,...,...,...,...,...,...
861,Toyota Truck 2WD,21.0,4.0,146.5,NaN,NaN,NaN,87,Japan,epa_public
1129,Toyota Truck 2WD,20.0,4.0,146.5,NaN,NaN,NaN,89,Japan,epa_public
1131,Toyota Truck 2WD,20.0,4.0,146.5,NaN,NaN,NaN,89,Japan,epa_public
1338,Volkswagen Quantum,31.0,4.0,97.6,NaN,NaN,NaN,84,Europe,epa_public


After dropping duplicates: (1358, 10)


In [19]:
# Show what dropping would cost (these return copies; self.df is unchanged)
print("Drop rows with any NaN:   ", cars.drop_missing(axis=0).shape)
print("Drop columns with any NaN:", cars.drop_missing(axis=1).shape)

# Compare simple strategies on a copy
for s in ["mean", "median", "mode"]:
    print(s, "→ MPG mean after fill:", round(cars.impute("MPG", strategy=s)["MPG"].mean(), 2))

Drop rows with any NaN:    (387, 10)
Drop columns with any NaN: (1358, 5)
mean → MPG mean after fill: 20.73
median → MPG mean after fill: 20.72
mode → MPG mean after fill: 20.71


In [20]:
for col in ["MPG", "Displacement", "Horsepower", "Weight"]:
    print(col, cars.center_stats(col).to_dict())

cars.distribution_plot("Horsepower").show()
cars.distribution_plot("MPG").show()

MPG {'mean': 20.727, 'median': 20.0, 'skew': 0.861}
Displacement {'mean': 193.189, 'median': 152.6, 'skew': 0.755}
Horsepower {'mean': 104.698, 'median': 95.0, 'skew': 1.049}
Weight {'mean': 2978.294, 'median': 2815.0, 'skew': 0.509}


In [21]:
# MPG depends strongly on engine size, so fill it within each cylinder group
cars.group_impute("MPG", by="Cylinders", strategy="median")

# Weight first, then Horsepower (which uses Weight as a predictor)
cars.regression_impute("Weight", ["Displacement", "Cylinders"])
cars.regression_impute("Horsepower", ["Displacement", "Cylinders", "Weight"])

# Whatever is left (e.g. Acceleration) gets the column median
cars.simple_impute(["Acceleration", "Displacement", "Horsepower", "Weight"], strategy="median")

display(cars.missing_report())

,missing,missing_%,valid
Car,0,0.0,1358
MPG,0,0.0,1358
Cylinders,0,0.0,1358
Displacement,0,0.0,1358
Horsepower,0,0.0,1358
Weight,0,0.0,1358
Acceleration,0,0.0,1358
Model,0,0.0,1358
Origin,0,0.0,1358
source,0,0.0,1358


In [22]:
display(cars.compare_before_after())
cars.before_after_plot("Horsepower").show()

display(cars.save_clean())

,before,after
rows,1406.00,1358.00
missing_cells,3021.00,0.00
duplicate_rows,48.00,0.00
MPG_mean,20.65,20.69
MPG_median,20.00,20.00
MPG_std,6.23,6.27
Displacement_mean,194.26,193.16
Displacement_median,152.60,152.60
Displacement_std,93.10,92.99
Horsepower_mean,104.70,104.27


,table,rows_written
0,cars_clean,1358
